# Dataset v2: Frozen ImageNet ResNet18 + BiGRU + balanced sampler

This notebook repeats the strongest dataset-v1 baseline on dataset v2.

Controlled comparison:

- grayscale mouth ROI, 24 frames, 96x96;
- frozen ImageNet-pretrained ResNet18 frame encoder;
- trainable BiGRU temporal classifier;
- speaker-word balanced sampling;
- the same validation speaker as in dataset v1;
- model selection by validation macro-F1;
- test split remains untouched.

Upload `colab_lipreading_dataset_v2.zip` to Google Drive, enable a GPU runtime,
and run the notebook from top to bottom.


## 1. Подключить Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Найти и распаковать датасет

In [ ]:
import zipfile
from pathlib import Path
import shutil

extract_dir = Path('/content/lipreading_data_v2')
relative_split_dir = (
    Path('03_splits')
    / 'ml_splits_mouth_crops_padded_dataset_v2'
    / 'speaker_top10'
)
expected_split_dir = extract_dir / 'ru_dataset' / relative_split_dir

if expected_split_dir.exists():
    print('Dataset v2 already exists:', extract_dir)
else:
    print('Dataset v2 not found, searching zip on Google Drive...')
    zip_names = [
        'colab_lipreading_dataset_v2.zip',
        'kaggle_lipreading_dataset_v2.zip',
    ]
    matches = []
    for zip_name in zip_names:
        matches.extend(Path('/content/drive').rglob(zip_name))

    print('matches:', matches)
    if not matches:
        raise FileNotFoundError(
            'Upload colab_lipreading_dataset_v2.zip to Google Drive first.'
        )

    zip_path = matches[0]
    print('zip:', zip_path)
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    print('Dataset v2 restored:', extract_dir)

data_root = extract_dir / 'ru_dataset'
split_dir = data_root / relative_split_dir
print('data_root:', data_root)
print('split exists:', split_dir.exists())


## 3. Настройки, split и vocabulary

In [ ]:
from pathlib import Path
from collections import Counter
import random
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler


def set_reproducible_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


SEED = 42
set_reproducible_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

if not (split_dir / 'train.csv').exists():
    raise FileNotFoundError(f'Dataset v2 split was not found: {split_dir}')

train_df = pd.read_csv(split_dir / 'train.csv')
val_df = pd.read_csv(split_dir / 'val.csv')
test_df = pd.read_csv(split_dir / 'test.csv')
vocab = (split_dir / 'vocab.txt').read_text(encoding='utf-8').splitlines()
word_to_idx = {word: index for index, word in enumerate(vocab)}
idx_to_word = {index: word for word, index in word_to_idx.items()}

checkpoint_dir = Path(
    '/content/drive/MyDrive/lipreading_sem4/controlled_frozen_resnet18_bigru_balanced_v2'
)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

missing_paths = [path for path in train_df['clip_path'] if not (data_root / path).is_file()]
missing_paths += [path for path in val_df['clip_path'] if not (data_root / path).is_file()]
assert not missing_paths, f'Missing clips: {len(missing_paths)}'
assert set(train_df['speaker_id']).isdisjoint(set(val_df['speaker_id']))
assert set(train_df['speaker_id']).isdisjoint(set(test_df['speaker_id']))
assert set(val_df['speaker_id']).isdisjoint(set(test_df['speaker_id']))

print('device:', device)
print('train:', len(train_df), 'speakers:', sorted(train_df['speaker_id'].unique()))
print('validation:', len(val_df), 'speakers:', sorted(val_df['speaker_id'].unique()))
print('test reserved:', len(test_df), 'speakers:', sorted(test_df['speaker_id'].unique()))
print('vocab:', vocab)
print('missing paths:', len(missing_paths))
print('results directory:', checkpoint_dir)


## 4. Распределение классов и majority baseline

In [ ]:
distribution_rows = []
for word in vocab:
    distribution_rows.append({
        'word': word,
        'train': int((train_df['word'] == word).sum()),
        'validation': int((val_df['word'] == word).sum()),
        'test_reserved': int((test_df['word'] == word).sum()),
    })

distribution_df = pd.DataFrame(distribution_rows)
display(distribution_df)

majority_word = val_df['word'].value_counts().index[0]
majority_baseline_acc = val_df['word'].value_counts().iloc[0] / len(val_df)
print('validation majority baseline:', majority_word, round(majority_baseline_acc, 4))
print('A useful model must beat this baseline without predicting only one word.')

## 5. Загрузка grayscale-видео

In [ ]:
def load_video(path, num_frames=24, size=96, grayscale=False):
    cap = cv2.VideoCapture(str(path))

    frames = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        raise ValueError(f'Cannot read video: {path}')

    frame_indices = np.linspace(0, total_frames - 1, num_frames).astype(int)

    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()

        if not ok:
            continue

        if grayscale:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            frame = cv2.resize(frame, (size, size))
            frame = frame[:, :, None]
        else:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (size, size))

        frame = frame / 255.0
        frames.append(frame)

    cap.release()

    if len(frames) == 0:
        raise ValueError(f'No frames loaded: {path}')

    while len(frames) < num_frames:
        frames.append(frames[-1])

    video = np.stack(frames)
    video = torch.tensor(video, dtype=torch.float32)
    video = video.permute(3, 0, 1, 2)

    return video

## 6. Video Dataset для однократного извлечения признаков

In [ ]:
class LipReadingDataset(Dataset):
    def __init__(self, df, data_root, word_to_idx):
        self.df = df.reset_index(drop=True)
        self.data_root = data_root
        self.word_to_idx = word_to_idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        video_path = self.data_root / row['clip_path']
        video = load_video(video_path, grayscale=True)
        label = torch.tensor(self.word_to_idx[row['word']], dtype=torch.long)
        return video, label


train_video_dataset = LipReadingDataset(train_df, data_root, word_to_idx)
val_video_dataset = LipReadingDataset(val_df, data_root, word_to_idx)

train_video_loader = DataLoader(
    train_video_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
)
val_video_loader = DataLoader(
    val_video_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
)

sample_videos, sample_labels = next(iter(train_video_loader))
print('sample videos:', sample_videos.shape)
print('sample labels:', sample_labels.tolist())

## 7. Frozen ImageNet ResNet18

Grayscale-кадр повторяется в три канала и нормализуется как ImageNet. BatchNorm внутри ResNet всегда остаётся в evaluation mode.

In [ ]:
import torchvision.models as tv_models


class FrozenImageNetResNet18(nn.Module):
    """Convert grayscale frames to fixed 512-dimensional ImageNet features."""

    def __init__(self):
        super().__init__()
        try:
            weights = tv_models.ResNet18_Weights.DEFAULT
            encoder = tv_models.resnet18(weights=weights)
            mean = weights.transforms().mean
            std = weights.transforms().std
        except AttributeError:
            encoder = tv_models.resnet18(pretrained=True)
            mean = [0.485, 0.456, 0.406]
            std = [0.229, 0.224, 0.225]

        encoder.fc = nn.Identity()
        for parameter in encoder.parameters():
            parameter.requires_grad = False

        self.encoder = encoder.eval()
        self.register_buffer('mean', torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(std).view(1, 3, 1, 1))

    def train(self, mode=True):
        super().train(mode)
        self.encoder.eval()
        return self

    def forward(self, videos):
        # videos: [batch, 1, frames, height, width]
        batch_size, _, frames, height, width = videos.shape
        images = videos.permute(0, 2, 1, 3, 4)
        images = images.reshape(batch_size * frames, 1, height, width)
        images = images.repeat(1, 3, 1, 1)
        images = (images - self.mean) / self.std

        with torch.no_grad():
            features = self.encoder(images)
        return features.reshape(batch_size, frames, -1)


frame_encoder = FrozenImageNetResNet18().to(device).eval()
with torch.no_grad():
    sample_features = frame_encoder(sample_videos.to(device))

print('sample frozen features:', sample_features.shape)
print('feature mean/std:', sample_features.mean().item(), sample_features.std().item())

## 8. Извлечь и сохранить frame features

Первый запуск создаёт train/validation feature cache на Google Drive. Повторный запуск загрузит готовый cache. Test features не создаются.

In [ ]:
def load_torch_file(path, device='cpu'):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def extract_feature_cache(encoder, loader, metadata_df, cache_path):
    all_features = []
    all_labels = []
    encoder.eval()
    start = time.perf_counter()

    for batch_index, (videos, labels) in enumerate(loader, start=1):
        features = encoder(videos.to(device)).cpu()
        all_features.append(features)
        all_labels.append(labels)

        if batch_index == 1 or batch_index % 25 == 0:
            print(f'feature batches: {batch_index}/{len(loader)}')

    cache = {
        'features': torch.cat(all_features),
        'labels': torch.cat(all_labels),
        'clip_paths': metadata_df['clip_path'].tolist(),
        'words': metadata_df['word'].tolist(),
        'speakers': metadata_df['speaker_id'].tolist(),
        'feature_extraction_seconds': time.perf_counter() - start,
        'encoder': 'torchvision ResNet18 ImageNet DEFAULT, frozen',
        'input': 'grayscale repeated to RGB, ImageNet normalization, 24x96x96',
        'dataset_version': 'v2',
    }
    torch.save(cache, cache_path)
    print('saved feature cache:', cache_path)
    return cache


train_cache_path = checkpoint_dir / 'dataset_v2_train_resnet18_imagenet_features.pt'
val_cache_path = checkpoint_dir / 'dataset_v2_val_resnet18_imagenet_features.pt'
REBUILD_FEATURE_CACHE = False

if REBUILD_FEATURE_CACHE or not train_cache_path.exists():
    train_cache = extract_feature_cache(
        frame_encoder,
        train_video_loader,
        train_df,
        train_cache_path,
    )
else:
    train_cache = load_torch_file(train_cache_path)
    print('loaded existing train feature cache:', train_cache_path)

if REBUILD_FEATURE_CACHE or not val_cache_path.exists():
    val_cache = extract_feature_cache(
        frame_encoder,
        val_video_loader,
        val_df,
        val_cache_path,
    )
else:
    val_cache = load_torch_file(val_cache_path)
    print('loaded existing validation feature cache:', val_cache_path)

assert len(train_cache['features']) == len(train_df)
assert len(val_cache['features']) == len(val_df)
assert train_cache['features'].shape[1:] == (24, 512)
assert val_cache['features'].shape[1:] == (24, 512)

print('train cached features:', train_cache['features'].shape)
print('validation cached features:', val_cache['features'].shape)
print('feature extraction minutes:', round(
    (train_cache['feature_extraction_seconds'] + val_cache['feature_extraction_seconds']) / 60,
    2,
))


## 9. Balanced DataLoader для последовательностей признаков

In [ ]:
class FeatureSequenceDataset(Dataset):
    def __init__(self, cache):
        self.features = cache['features'].float()
        self.labels = cache['labels'].long()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]


# Each non-empty (speaker, word) group receives equal total sampling probability.
group_keys = list(zip(train_cache['speakers'], train_cache['words']))
group_counts = Counter(group_keys)
sample_weights = torch.tensor(
    [1.0 / group_counts[key] for key in group_keys],
    dtype=torch.double,
)
sampler_generator = torch.Generator().manual_seed(SEED)
balanced_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
    generator=sampler_generator,
)

train_feature_dataset = FeatureSequenceDataset(train_cache)
val_feature_dataset = FeatureSequenceDataset(val_cache)
train_loader = DataLoader(
    train_feature_dataset,
    batch_size=32,
    sampler=balanced_sampler,
    num_workers=0,
)
val_loader = DataLoader(
    val_feature_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

print('non-empty speaker-word groups:', len(group_counts))
print('sample weight min/max:', sample_weights.min().item(), sample_weights.max().item())
feature_batch, label_batch = next(iter(train_loader))
print('feature batch:', feature_batch.shape)
print('label batch:', label_batch.shape)


## 10. Trainable BiGRU classifier

In [ ]:
class BiGRUClassifier(nn.Module):
    def __init__(self, num_classes, input_dim=512, hidden_dim=128, dropout=0.4):
        super().__init__()
        self.temporal_encoder = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, num_classes),
        )

    def forward(self, features):
        _, hidden = self.temporal_encoder(features)
        clip_features = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(clip_features)


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


set_reproducible_seed(SEED)
model = BiGRUClassifier(num_classes=len(vocab)).to(device)
print('trainable BiGRU/classifier parameters:', count_parameters(model))
print('frozen ResNet18 parameters:', sum(p.numel() for p in frame_encoder.parameters()))

with torch.no_grad():
    logits = model(feature_batch.to(device))
print('logits:', logits.shape)

## 11. Метрики и controlled training loop

In [ ]:
def confusion_matrix_array(preds, labels, num_classes):
    matrix = np.zeros((num_classes, num_classes), dtype=int)
    for true_index, pred_index in zip(labels, preds):
        matrix[true_index, pred_index] += 1
    return matrix


def classification_metrics(preds, labels, num_classes):
    matrix = confusion_matrix_array(preds, labels, num_classes)
    total = int(matrix.sum())
    accuracy = float(np.trace(matrix) / total) if total else 0.0
    class_rows = []

    for class_id in range(num_classes):
        true_positive = int(matrix[class_id, class_id])
        support = int(matrix[class_id, :].sum())
        predicted_count = int(matrix[:, class_id].sum())
        precision = true_positive / predicted_count if predicted_count else 0.0
        recall = true_positive / support if support else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        class_rows.append({
            'class_id': class_id,
            'support': support,
            'predicted_count': predicted_count,
            'correct': true_positive,
            'precision': precision,
            'recall': recall,
            'f1': f1,
        })

    active_rows = [row for row in class_rows if row['support'] > 0]
    macro_f1 = float(np.mean([row['f1'] for row in active_rows])) if active_rows else 0.0
    balanced_accuracy = float(np.mean([row['recall'] for row in active_rows])) if active_rows else 0.0
    return {
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'balanced_accuracy': balanced_accuracy,
        'class_rows': class_rows,
        'confusion_matrix': matrix,
    }


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)
        correct += (outputs.argmax(dim=1) == batch_y).sum().item()
        total += batch_y.size(0)

    return total_loss / total, correct / total


def evaluate_detailed(model, loader, criterion, device, num_classes):
    model.eval()
    total_loss = 0.0
    total = 0
    preds = []
    labels = []

    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            total_loss += loss.item() * batch_x.size(0)
            total += batch_y.size(0)
            preds.extend(outputs.argmax(dim=1).cpu().tolist())
            labels.extend(batch_y.cpu().tolist())

    metrics = classification_metrics(preds, labels, num_classes)
    metrics['loss'] = total_loss / total
    metrics['preds'] = preds
    metrics['labels'] = labels
    return metrics


def load_checkpoint(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def fit_controlled_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    num_epochs=30,
    patience=7,
):
    best_checkpoint_path = checkpoint_dir / 'frozen_resnet18_bigru_balanced_v2_best.pt'
    history_path = checkpoint_dir / 'frozen_resnet18_bigru_balanced_v2_history.csv'
    history_rows = []
    best_macro_f1 = -1.0
    best_val_loss = float('inf')
    epochs_without_improvement = 0
    total_start = time.perf_counter()

    for epoch in range(1, num_epochs + 1):
        epoch_start = time.perf_counter()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_metrics = evaluate_detailed(model, val_loader, criterion, device, len(vocab))
        epoch_seconds = time.perf_counter() - epoch_start

        row = {
            'epoch': epoch,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_metrics['loss'],
            'val_acc': val_metrics['accuracy'],
            'val_macro_f1': val_metrics['macro_f1'],
            'val_balanced_acc': val_metrics['balanced_accuracy'],
            'epoch_seconds': epoch_seconds,
        }
        history_rows.append(row)
        pd.DataFrame(history_rows).to_csv(history_path, index=False)

        print(
            f"epoch {epoch}/{num_epochs} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['accuracy']:.4f} "
            f"val_macro_f1={val_metrics['macro_f1']:.4f} "
            f"val_bal_acc={val_metrics['balanced_accuracy']:.4f} | {epoch_seconds:.1f}s"
        )

        improved = (
            val_metrics['macro_f1'] > best_macro_f1
            or (
                val_metrics['macro_f1'] == best_macro_f1
                and val_metrics['loss'] < best_val_loss
            )
        )
        if improved:
            best_macro_f1 = val_metrics['macro_f1']
            best_val_loss = val_metrics['loss']
            epochs_without_improvement = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'vocab': vocab,
                'word_to_idx': word_to_idx,
                'idx_to_word': idx_to_word,
                'best_val_macro_f1': best_macro_f1,
                'best_val_acc': val_metrics['accuracy'],
                'best_val_balanced_accuracy': val_metrics['balanced_accuracy'],
                'best_val_loss': best_val_loss,
                'trainable_parameters': count_parameters(model),
                'history': history_rows,
            }, best_checkpoint_path)
            print('saved best by validation macro-F1:', best_checkpoint_path)
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f'early stop: no macro-F1 improvement for {patience} epochs')
            break

    total_seconds = time.perf_counter() - total_start
    return pd.DataFrame(history_rows), best_checkpoint_path, history_path, total_seconds

## 12. Обучение BiGRU

Максимум 30 эпох, AdamW, early stopping с patience 7, checkpoint по validation macro-F1.

In [ ]:
set_reproducible_seed(SEED)
model = BiGRUClassifier(num_classes=len(vocab)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

history_df, best_checkpoint_path, history_path, total_training_seconds = fit_controlled_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=30,
    patience=7,
)

print('epochs completed:', len(history_df))
print('BiGRU training minutes:', round(total_training_seconds / 60, 2))
print('history:', history_path)

## 13. Оценить лучший checkpoint на validation

In [ ]:
checkpoint = load_torch_file(best_checkpoint_path, device=device)
best_model = BiGRUClassifier(num_classes=len(vocab)).to(device)
best_model.load_state_dict(checkpoint['model_state_dict'])

val_metrics = evaluate_detailed(best_model, val_loader, criterion, device, len(vocab))
print('best epoch:', checkpoint['epoch'])
print('validation accuracy:', round(val_metrics['accuracy'], 4))
print('validation macro-F1:', round(val_metrics['macro_f1'], 4))
print('validation balanced accuracy:', round(val_metrics['balanced_accuracy'], 4))
print('majority baseline accuracy:', round(majority_baseline_acc, 4))

class_report_rows = []
for row in val_metrics['class_rows']:
    class_report_rows.append({
        'word': idx_to_word[row['class_id']],
        'support': row['support'],
        'predicted_count': row['predicted_count'],
        'correct': row['correct'],
        'precision': row['precision'],
        'recall': row['recall'],
        'f1': row['f1'],
    })

class_report_df = pd.DataFrame(class_report_rows)
display(class_report_df)

prediction_counts = Counter(idx_to_word[index] for index in val_metrics['preds'])
print('predicted word distribution:')
print(pd.Series(prediction_counts).sort_values(ascending=False))

validation_predictions_df = val_df.copy()
validation_predictions_df['true_id'] = val_metrics['labels']
validation_predictions_df['pred_id'] = val_metrics['preds']
validation_predictions_df['pred_word'] = [idx_to_word[index] for index in val_metrics['preds']]
validation_predictions_df['correct'] = (
    validation_predictions_df['true_id'] == validation_predictions_df['pred_id']
)

class_report_path = checkpoint_dir / 'balanced_v2_class_report.csv'
predictions_path = checkpoint_dir / 'balanced_v2_val_predictions.csv'
class_report_df.to_csv(class_report_path, index=False)
validation_predictions_df.to_csv(predictions_path, index=False)
print('saved class report:', class_report_path)
print('saved validation predictions:', predictions_path)


## 14. Графики и confusion matrix

In [ ]:
curves_path = checkpoint_dir / 'balanced_v2_training_curves.png'
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train')
axes[0].plot(history_df['epoch'], history_df['val_loss'], label='validation')
axes[0].set_title('Loss')
axes[0].set_xlabel('epoch')
axes[0].legend()

axes[1].plot(history_df['epoch'], history_df['train_acc'], label='train accuracy')
axes[1].plot(history_df['epoch'], history_df['val_acc'], label='validation accuracy')
axes[1].axhline(majority_baseline_acc, color='gray', linestyle='--', label='majority baseline')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('epoch')
axes[1].legend()

axes[2].plot(history_df['epoch'], history_df['val_macro_f1'], label='macro-F1')
axes[2].plot(history_df['epoch'], history_df['val_balanced_acc'], label='balanced accuracy')
axes[2].set_title('Balanced validation metrics')
axes[2].set_xlabel('epoch')
axes[2].legend()

fig.tight_layout()
fig.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.show()
print('saved curves:', curves_path)

matrix = val_metrics['confusion_matrix']
confusion_path = checkpoint_dir / 'balanced_v2_confusion_matrix.png'
fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(matrix, cmap='Blues')
ax.set_title('Dataset v2: Frozen ResNet18 + BiGRU + balanced sampler')
ax.set_xlabel('predicted word')
ax.set_ylabel('true word')
ax.set_xticks(range(len(vocab)), vocab, rotation=45, ha='right')
ax.set_yticks(range(len(vocab)), vocab)
fig.colorbar(image, ax=ax)
for true_index in range(matrix.shape[0]):
    for pred_index in range(matrix.shape[1]):
        if matrix[true_index, pred_index]:
            ax.text(pred_index, true_index, str(matrix[true_index, pred_index]), ha='center', va='center')
fig.tight_layout()
fig.savefig(confusion_path, dpi=150, bbox_inches='tight')
plt.show()
print('saved confusion matrix:', confusion_path)

## 15. Сравнение dataset v2 с balanced baseline на dataset v1

In [ ]:
predicted_class_count = len(set(val_metrics['preds']))
balanced_v2_summary = pd.DataFrame([{
    'model': 'Frozen ImageNet ResNet18 features + BiGRU + speaker-word sampler',
    'dataset_version': 'v2',
    'split': 'speaker-based',
    'train_clips': len(train_df),
    'validation_clips': len(val_df),
    'best_epoch': checkpoint['epoch'],
    'validation_accuracy': val_metrics['accuracy'],
    'validation_macro_f1': val_metrics['macro_f1'],
    'validation_balanced_accuracy': val_metrics['balanced_accuracy'],
    'majority_baseline_accuracy': majority_baseline_acc,
    'predicted_classes': predicted_class_count,
    'trainable_parameters': checkpoint['trainable_parameters'],
    'training_minutes': total_training_seconds / 60,
    'test_used': False,
}])

summary_path = checkpoint_dir / 'balanced_v2_summary.csv'
balanced_v2_summary.to_csv(summary_path, index=False)
display(balanced_v2_summary)

v1_summary_path = Path(
    '/content/drive/MyDrive/lipreading_sem4/'
    'controlled_frozen_resnet18_bigru_balanced/balanced_summary.csv'
)
comparison_frames = []
if v1_summary_path.exists():
    v1_summary = pd.read_csv(v1_summary_path)
    if 'dataset_version' not in v1_summary.columns:
        v1_summary['dataset_version'] = 'v1'
    comparison_frames.append(v1_summary)
else:
    print('Dataset v1 balanced summary was not found:', v1_summary_path)

comparison_frames.append(balanced_v2_summary)
comparison = pd.concat(comparison_frames, ignore_index=True)
comparison_columns = [
    'dataset_version',
    'model',
    'train_clips',
    'validation_accuracy',
    'validation_macro_f1',
    'validation_balanced_accuracy',
    'predicted_classes',
    'test_used',
]
display(comparison[[column for column in comparison_columns if column in comparison.columns]])

print('saved summary:', summary_path)
print('TEST SPLIT WAS NOT USED.')


## How to interpret this experiment

Compare dataset v2 with the earlier balanced dataset-v1 run using validation
macro-F1 as the main metric. Validation and test speakers are unchanged, so a
higher v2 score indicates that adding two training speakers improved
speaker-independent generalization. Do not open the test split until the final
model and protocol have been selected.
